# Which observations are available?
Inspect three Swiss stations in NOAA ISD-Lite during 2024. Start with the station catalogue, then count finite observations on the common hourly grid. Run the cells from top to bottom after installing the notebook environment described in the README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from giano.data import load_input

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sample = root / "examples" / "noaa-2024"
torch.set_num_threads(2)
ds = load_input(sample / "observations.csv", sample / "stations.csv")

In [ ]:
catalogue = pd.read_csv(sample / "stations.csv", dtype={"station_id": str})
display(catalogue)
coverage = ds.count("time").to_dataframe()
coverage.div(ds.sizes["time"]).round(3)

## Seasonal coverage
Each point is the fraction of scheduled hours with a recorded temperature in that month. A station can have high annual coverage and still have a long continuous outage.

In [ ]:
monthly = ds.temperature.notnull().resample(time="MS").mean().to_pandas()
ax = monthly.plot(
    marker="o", ylim=(0, 1.05), ylabel="Fraction observed", xlabel="Month (UTC)"
)
ax.legend(title="Station")
plt.show()

## Interpreting a missing wind direction
At zero observed speed there is no physical direction. Count those hours separately from missing directions at positive speed and from hours when speed itself is missing.

In [ ]:
pd.DataFrame(
    {
        "calm and direction missing": (
            (ds.wind_speed == 0) & ds.wind_direction.isnull()
        )
        .sum("time")
        .to_pandas(),
        "moving and direction missing": (
            (ds.wind_speed > 0) & ds.wind_direction.isnull()
        )
        .sum("time")
        .to_pandas(),
        "speed missing": ds.wind_speed.isnull().sum("time").to_pandas(),
    }
)

## Coverage in the original Meteotrentino network
The bundled table contains one row per measured station and variable in the historical explorer. Each instrument has its own first and last measurement. Sum the observed and scheduled hours before dividing to give each station-hour equal weight. Compare that with the median station coverage, which gives every station equal weight. Neither measure estimates the accuracy of gap filling.

In [ ]:
trentino = pd.read_csv(root / "examples" / "trentino" / "station_coverage.csv")
trentino["coverage"] = trentino.observed_hours / trentino.span_hours
by_variable = trentino.groupby("variable").agg(
    stations=("station", "nunique"),
    observed=("observed_hours", "sum"),
    span=("span_hours", "sum"),
    missing=("missing_hours", "sum"),
    median_station_coverage=("coverage", "median"),
)
by_variable["hour_weighted_coverage"] = by_variable.observed / by_variable.span
by_variable.round(3)

In [ ]:
ax = by_variable[["median_station_coverage", "hour_weighted_coverage"]].plot.barh(
    xlim=(0, 1.01),
    xlabel="Fraction observed",
    ylabel="",
    title="Meteotrentino instrument coverage",
)
ax.legend(["Median across stations", "Weighted by station-hours"])
plt.show()